# SAP-RPT-TS — zero-shot forecast example

SAP's time-series foundation model **SAP-RPT-TS** produces forecasts with no per-dataset
training. This notebook runs the same forecast two ways on a **synthetic** energy-consumption
dataset — a teaching example to show how each method is used, not a benchmark (the accuracy
numbers are illustrative):

- **§1 — SAP HANA PAL** via the built-in **TSFM** pipeline operator (`hana_ml` client).
- **§2 — SAP AI Core REST** via the model's `/predict` endpoint.

> **Scope:** 2026 Q3 release. **Single-target** (univariate) forecasting — one target series
> per call, with optional past and future covariates. Multi-target is not supported.

## Import packages

This cell imports everything used by **both** flows, so you can run either one on its own.
Each flow then loads the dataset itself — §1 (SAP HANA PAL) and §2 (SAP AI Core REST) are
independent and share no state beyond these imports.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, force=True)

# Shared
import json
import pandas as pd
import numpy as np

# §1 - SAP HANA PAL
from hana_ml import ConnectionContext
from hana_ml.dataframe import create_dataframe_from_pandas
from hana_ml.algorithms.pal.pipeline import Pipeline
from hana_ml.visualizers.visualizer_base import forecast_line_plot

# §2 - SAP AI Core REST
from pathlib import Path
from ai_api_client_sdk.ai_api_v2_client import AIAPIV2Client
import requests

## 1. SAP HANA PAL

**hana-ml** is the Python machine-learning client for SAP HANA: it lets you access HANA data
and build ML models on it directly in the database. SAP-RPT-TS is exposed through its built-in
**TSFM** pipeline operator — we connect to HANA, upload the prepared table, and call
`PAL_PIPELINE_PREDICT` in skip-fit mode (no separate fit step; the model forecasts directly
from the context it receives).

```bash
pip install hana-ml ai-api-client-sdk   # https://pypi.org/project/hana-ml/
```

### 1.1 Load data

Daily energy consumption for a commercial building over one year (`ts_energy_consumption.csv`,
365 rows). Target is `energy_consumption_kwh`; covariates split into:

| Type | Columns | In future rows? |
|---|---|---|
| Future | `temperature_c`, `humidity_pct`, `solar_irradiance_kwh_m2`, `wind_speed_ms`, `is_weekend`, `is_holiday` | ✓ populated |
| Past | `occupancy_rate` | ✗ set to `NULL` |

PAL takes one table holding history + horizon: we hold out the last `HORIZON` days (target →
`NULL`) so TSFM forecasts them, keeping the true values to score against.

In [ ]:
CSV_PATH = 'ts_energy_consumption.csv'   # shipped alongside this notebook
HORIZON = 14   # forecast the last 14 days

TARGET = 'energy_consumption_kwh'
FUTURE_COVARIATES = ['temperature_c', 'humidity_pct', 'solar_irradiance_kwh_m2',
                     'wind_speed_ms', 'is_weekend', 'is_holiday']   # known over the horizon
PAST_COVARIATES   = ['occupancy_rate']                              # known only in history
COVARIATES = FUTURE_COVARIATES + PAST_COVARIATES

df = pd.read_csv(CSV_PATH, parse_dates=['date']).sort_values('date').reset_index(drop=True)
df.insert(0, 'ID', np.arange(1, len(df) + 1))   # integer key required by the operator

# Keep the true future target so we can score afterwards, then blank it for the model.
actuals = df[TARGET].copy()

input_df = df[['ID', TARGET] + COVARIATES].copy()
future_mask = input_df['ID'] > len(df) - HORIZON

input_df.loc[future_mask, TARGET] = np.nan            # NULL target => forecast rows
input_df.loc[future_mask, PAST_COVARIATES] = np.nan   # past covariate: unknown over horizon

print(f'History rows: {len(df) - HORIZON},  forecast horizon: {HORIZON}')
print(f'Future covariates: {FUTURE_COVARIATES}')
print(f'Past covariates:   {PAST_COVARIATES}')
input_df.tail(HORIZON + 2)

### 1.2 Connect to HANA

Uses an `hdbuserstore` key so no password appears in the notebook. Create one from a terminal
(`hdbuserstore SET <KEY> <host>:<port> <user> <password>`) and set its name below, or connect
explicitly with `ConnectionContext(address=..., port=..., user=..., password=..., encrypt=True)`.

In [ ]:
conn = ConnectionContext(userkey='<your-hdbuserstore-key>')
print('HANA version:', conn.hana_version())

### 1.3 Upload data to HANA

Use `replace=False` so `NaN` is stored as **`NULL`** — with `replace=True`, `hana_ml` turns
`NULL` into `0`, the operator then sees no horizon and raises *"no future rows found"*.

In [ ]:
hdf = create_dataframe_from_pandas(
    conn,
    input_df,
    table_name='TSFM_ENERGY_TBL',
    force=True,            # drop+recreate if it exists
    replace=False,         # keep NaN as NULL (True would turn NULL into 0)
)
print(hdf.count(), 'dataset uploaded')
hdf.head(3).collect()

Verify what HANA actually stores — first rows of history and the horizon tail where target and past covariate are `NULL`:

In [ ]:
# Read the table straight from HANA (a fresh server-side query, not the local frame).
hana_table = conn.table('TSFM_ENERGY_TBL')
print('Columns  :', hana_table.columns)
print('Row count:', hana_table.count())

print('\n-- first 5 rows --')
display(hana_table.head(5).collect())

# Forecast horizon: TARGET and the past covariate are NULL here.
print('-- last {} rows (forecast horizon, TARGET = NULL) --'.format(HORIZON))
display(hana_table.sort('ID', desc=True).head(HORIZON).collect())

### 1.4 Run zero-shot forecast

Build a `Pipeline` from the TSFM operator JSON and call `predict`; with no fitted model it
auto-selects skip-fit mode. `output_prediction_interval=1` (sets `OUTPUT_QUANTILE=1`
internally) returns each row's `SCORES` as JSON — `{"value", "lower", "upper"}` — i.e. the
point forecast plus bounds at the requested `confidence_level`.

In [ ]:
pipe = Pipeline(pipeline={'TSFM': {'args': {}, 'inputs': {'data': 'ROWDATA'}}})

CONFIDENCE = 0.9   # prediction-interval confidence level

result = pipe.predict(
    data=hdf,
    key='ID',
    exog=COVARIATES,   # future covariates populated over the horizon; past covariate NULL there
    skip_fit=True,
    output_prediction_interval=1,   # -> OUTPUT_QUANTILE=1, returns the interval
    confidence_level=CONFIDENCE,
)

# With prediction intervals, SCORES is a JSON string: {"value":..,"lower":..,"upper":..}.
raw = result.collect()
parsed = raw['SCORES'].apply(json.loads).apply(pd.Series)
forecast = pd.concat([raw['ID'].astype(int), parsed], axis=1)
forecast = forecast.rename(columns={'value': 'forecast'})
forecast[['forecast', 'lower', 'upper']] = forecast[['forecast', 'lower', 'upper']].astype(float)
forecast

### 1.5 Results

Point accuracy (MAE, MAPE), prediction-interval coverage, per-day table, and chart.

In [ ]:
cmp = forecast.merge(
    df[['ID', 'date']].assign(actual=actuals),
    on='ID', how='left',
)

mae = (cmp['forecast'] - cmp['actual']).abs().mean()
mape = ((cmp['forecast'] - cmp['actual']).abs() / cmp['actual']).mean() * 100
coverage = ((cmp['actual'] >= cmp['lower']) & (cmp['actual'] <= cmp['upper'])).mean() * 100
print(f'MAE  = {mae:.2f} kWh')
print(f'MAPE = {mape:.2f} %')
print(f'{CONFIDENCE:.0%} interval coverage = {coverage:.1f} % of actuals inside [lower, upper]')

cmp

In [ ]:
# forecast_line_plot wants two HANA DataFrames with a time index: the prediction frame
# (point forecast + bound columns) and the actual frame (observed series).
pred_pdf = cmp[['date', 'forecast', 'lower', 'upper']].rename(
    columns={'date': 'TS', 'forecast': 'FORECAST', 'lower': 'LOWER', 'upper': 'UPPER'})
actual_pdf = df[['date', TARGET]].rename(columns={'date': 'TS', TARGET: 'ACTUAL'})

pred_hdf = create_dataframe_from_pandas(
    conn, pred_pdf, 'TSFM_PRED_PLOT', force=True, replace=False).set_index('TS')
actual_hdf = create_dataframe_from_pandas(
    conn, actual_pdf, 'TSFM_ACTUAL_PLOT', force=True, replace=False).set_index('TS')

ax = forecast_line_plot(
    pred_data=pred_hdf,
    actual_data=actual_hdf,
    confidence=('LOWER', 'UPPER'),          # shaded prediction-interval band
    enable_plotly=False,                    # matplotlib (clean white-background screenshot)
    figsize=(12, 6),
    actual_option={'zorder': 1, 'alpha': 0.6},
)
ax.set_title(f'SAP-RPT-TS forecast ({CONFIDENCE:.0%} interval)')
ax.set_ylabel('kWh')
ax.figure.set_facecolor('white')       # white background for the screenshot
ax.set_facecolor('white')
ax.figure                              # render the figure inline

In [ ]:
conn.close()

## 2. SAP AI Core REST

SAP-RPT-TS is also served via **Generative AI Hub in SAP AI Core** (model `sap-rpt-ts`,
executable `aicore-sap`). The interface is a single `/predict` HTTP endpoint — no HANA
connection required. This section is self-contained: it reloads the dataset and does not depend
on §1 (only on the shared imports at the top).

API reference: https://help.sap.com/docs/sap-ai-core/generative-ai/sap-rpt-ts

### 2.1 Load data

Load the same CSV and split it into the history window and the forecast horizon. The REST API
takes the raw history plus known-over-horizon covariates directly — no HANA table and no
`NULL` padding.

In [ ]:
CSV_PATH = 'ts_energy_consumption.csv'
HORIZON = 14          # forecast the last 14 days
CONFIDENCE = 0.9      # prediction-interval confidence level

TARGET = 'energy_consumption_kwh'
FUTURE_COVARIATES = ['temperature_c', 'humidity_pct', 'solar_irradiance_kwh_m2',
                     'wind_speed_ms', 'is_weekend', 'is_holiday']   # known over the horizon
PAST_COVARIATES   = ['occupancy_rate']                              # known only in history
COVARIATES = FUTURE_COVARIATES + PAST_COVARIATES

df = pd.read_csv(CSV_PATH, parse_dates=['date']).sort_values('date').reset_index(drop=True)
hist = df.iloc[: len(df) - HORIZON]     # history
fut  = df.iloc[len(df) - HORIZON :]     # forecast horizon

print(f'History rows: {len(hist)},  forecast horizon: {HORIZON}')
df.tail(HORIZON + 2)

### 2.2 Authenticate

The service key is `credential-type: x509`. Pass its `certificate` and `key` PEM strings
directly to `AIAPIV2Client` — the SDK handles mTLS token exchange and refresh internally.
Set `KEY_FILE` to your service-key JSON and `DEPLOYMENT_URL` to your deployment;
keep the key file out of version control.

In [ ]:
KEY_FILE       = "service_key.json"        # path to your AI Core service-key JSON
DEPLOYMENT_URL = "<your-deployment-url>"   # .../v2/inference/deployments/<id>
RESOURCE_GROUP = "default"

k = json.loads(Path(KEY_FILE).read_text())
client = AIAPIV2Client(
    base_url=k["serviceurls"]["AI_API_URL"] + "/v2",
    auth_url=k["certurl"].rstrip("/") + "/oauth/token",
    client_id=k["clientid"],
    cert_str=k["certificate"],
    key_str=k["key"],
    resource_group=RESOURCE_GROUP,
)
_predict_path = DEPLOYMENT_URL.split("/v2", 1)[-1].rstrip("/") + "/predict"
print("AI Core client ready.")

### 2.3 Build payload

The `/predict` endpoint takes `horizon`, a `data` list of series objects, and an optional
`options` block. Each series needs `timestamp`, `target` (list of lists for univariate),
`past_covariates` (history length), and `future_covariates` (horizon length; every key must
also appear in `past_covariates`).

In [ ]:
# Per the AI Core docs:
#   target            : history values (list of lists -> one variate here)
#   past_covariates   : every covariate, over the history window
#   future_covariates : known-over-horizon covariates only; each key must also appear
#                       in past_covariates
payload = {
    "horizon": HORIZON,
    "data": [
        {
            "timestamp": hist["date"].dt.strftime("%Y-%m-%d").tolist(),
            "target": [hist[TARGET].tolist()],
            "past_covariates":   {c: hist[c].tolist() for c in COVARIATES},
            "future_covariates": {c: fut[c].tolist()  for c in FUTURE_COVARIATES},
        }
    ],
    "options": {"confidence_interval": CONFIDENCE},   # 0.9 -> q0.05 / q0.95 bounds
}
print(json.dumps(payload, indent=2)[:600], "...")

### 2.4 Call /predict and compare

In [ ]:
# POST /predict; the SDK sends the bearer token automatically.
out = client.rest_client.post(
    path=_predict_path,
    body=payload,
    convert_body_to_camel_case=False,
)

# result:                     [n_series][n_variates][horizon]
# confidence_interval_output: [n_series][n_variates][2][horizon]  (0 lower, 1 upper)
point  = out["result"][0][0]
lower  = out["confidence_interval_output"][0][0][0]
upper  = out["confidence_interval_output"][0][0][1]
stamps = out["result_timestamp"][0]

rest_forecast = pd.DataFrame(
    {"date": pd.to_datetime(stamps), "forecast": point, "lower": lower, "upper": upper}
)

# Compare against the held-out actuals, exactly like the PAL section.
rest_cmp = rest_forecast.merge(
    df[["date", TARGET]].rename(columns={TARGET: "actual"}), on="date", how="left"
)
mae  = (rest_cmp["forecast"] - rest_cmp["actual"]).abs().mean()
mape = ((rest_cmp["forecast"] - rest_cmp["actual"]).abs() / rest_cmp["actual"]).mean() * 100
coverage = ((rest_cmp["actual"] >= rest_cmp["lower"]) &
            (rest_cmp["actual"] <= rest_cmp["upper"])).mean() * 100
print(f"MAE  = {mae:.2f} kWh")
print(f"MAPE = {mape:.2f} %")
print(f"{CONFIDENCE:.0%} interval coverage = {coverage:.1f} %")
rest_cmp